# 02. 실습: Computational Buffer 시뮬레이션

목표: 의미 없는 더미 trace도 추가 계산 시간을 제공하면 회상 확률을 높일 수 있다는 아이디어를 toy model로 실험합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 회상 확률 함수 만들기

아래 함수는 더미 reasoning 길이가 어느 정도까지는 도움을 주지만, 너무 길면 문맥 오염과 분산으로 성능이 떨어지는 비단조 패턴을 흉내 냅니다.

In [ ]:
import math
import random


def recall_probability(base_prob, dummy_tokens, natural_trace=False):
    # 토큰이 늘수록 latent computation이 늘어나는 이득입니다.
    buffer_gain = 0.22 * (1 - math.exp(-dummy_tokens / 800))
    # 너무 긴 더미 trace는 노이즈와 문맥 부담을 만든다고 가정합니다.
    overthinking_penalty = 0.10 * max(0, (dummy_tokens - 2048) / 4096)
    # 자연 reasoning은 더미보다 factual content가 있어 추가 이득을 줍니다.
    semantic_gain = 0.12 if natural_trace else 0.0
    prob = base_prob + buffer_gain + semantic_gain - overthinking_penalty
    return max(0.0, min(0.95, prob))


for length in [0, 128, 512, 1024, 2048, 4096, 8192]:
    dummy = recall_probability(0.20, length)
    natural = recall_probability(0.20, length, natural_trace=True)
    print(f"tokens={length:4d} dummy={dummy:.3f} natural={natural:.3f}")

## 2. pass@k 몬테카를로 샘플링

각 샘플이 독립적으로 정답을 맞힐 확률을 `p`라고 단순화하면, 여러 개를 뽑을수록 pass@k가 올라갑니다. 실제 모델 샘플은 독립이 아니지만 직관을 보기에는 충분합니다.

In [ ]:
def simulated_pass_at_k(probability, k, trials=5000, seed=7):
    rng = random.Random(seed)
    successes = 0
    for _ in range(trials):
        if any(rng.random() < probability for _ in range(k)):
            successes += 1
    return successes / trials


conditions = [
    ("OFF", recall_probability(0.20, 0)),
    ("ON Single Dummy", recall_probability(0.20, 32)),
    ("ON Dummy 1024", recall_probability(0.20, 1024)),
    ("ON Dummy 4096", recall_probability(0.20, 4096)),
    ("ON Natural", recall_probability(0.20, 1024, natural_trace=True)),
]

print("condition | p | pass@1 | pass@10 | pass@50")
print("--- | --- | --- | --- | ---")
for name, p in conditions:
    print(
        f"{name} | {p:.3f} | {simulated_pass_at_k(p, 1):.3f} | "
        f"{simulated_pass_at_k(p, 10):.3f} | {simulated_pass_at_k(p, 50):.3f}"
    )

## 3. 더미 길이 sweep

논문의 메시지처럼, 더미 computation은 어느 지점까지는 도움이 되지만 길수록 항상 좋은 것은 아닙니다.

In [ ]:
lengths = [0, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]
best = None
for length in lengths:
    p = recall_probability(0.20, length)
    pass10 = simulated_pass_at_k(p, 10, trials=2000, seed=length + 3)
    row = {"length": length, "p": p, "pass10": pass10}
    if best is None or row["pass10"] > best["pass10"]:
        best = row
    print(f"length={length:5d} p={p:.3f} pass@10={pass10:.3f}")

print("best dummy length:", best)

## 4. 해석

더미 trace는 의미가 없지만 token 생성이 추가 계산 시간을 제공합니다. 그러나 자연 trace의 사실 내용까지 대체하지는 못합니다. 이것이 computational buffer와 factual priming을 분리해서 보아야 하는 이유입니다.